# Nova Workshop 2026
<br/>
<img src="https://www.polyestertime.com/wp-content/uploads/2017/01/Nova-Chemical-23-09-2016.jpg" />
<br/><br/>

## Introduction
Having completed our application back-end, we will now:

1. Build a `build_incident_context(line_id)` function that returns recent KPIs and incidents.
2. Call an external model/agent endpoint `nova-incident-analyst` (Model Serving) with that context and a question.
3. Display the structured response.

Goal:
- Prototype the logic for the "Incident Copilot" before wiring it into the Databricks App.


In [0]:
import pyspark.sql.functions as F

user_schema = "andrij_demo"          # TODO
catalog_name = "nova_workshop"
lakebase_db = "nova_incidents"

gold_table = f"{catalog_name}.{user_schema}.gold_daily_line_kpis"
incidents_table = f"{catalog_name}.{user_schema}.gold_incidents"

print("Gold KPIs table :", gold_table)
print("Incidents table :", incidents_table)

## Step 1 – Build Incident Context

We retrieve:
- Last 14 days of KPIs for the line.
- Last 20 incidents for the line.

The function will return a Python dict suitable for JSON serialization.

In [0]:
def build_incident_context(line_id: str, days: int = 14, max_incidents: int = 20):
    kpis_df = (
        spark.read.table(gold_table)
             .filter(F.col("line_id") == line_id)
             .orderBy(F.col("day").desc())
             .limit(days)
    )

    incidents_df = (
        spark.read.table(incidents_table)
             .filter(F.col("line_id") == line_id)
             .orderBy(F.col("ts").desc())
             .limit(max_incidents)
    )

    kpis = kpis_df.toPandas().to_dict(orient="records")
    incidents = incidents_df.toPandas().to_dict(orient="records")

    context = {
        "recent_kpis": kpis,
        "recent_incidents": incidents,
    }
    return context

# Quick sanity check
ctx = build_incident_context("LINE_01")
ctx.keys(), len(ctx["recent_kpis"]), len(ctx["recent_incidents"])

## Step 2 – Call Model/Agent Endpoint

Assumptions:
- A Model Serving endpoint (e.g. `nova-incident-analyst`) is deployed.
- URL and token are available via environment variables:
  - `INCIDENT_ANALYST_URL`
  - `INCIDENT_ANALYST_TOKEN`

We'll call the endpoint with JSON:
```json
{
  "question": "...",
  "context": {
    "recent_kpis": [...],
    "recent_incidents": [...]
  }
}


In [0]:
import os
import requests
import json

INCIDENT_ANALYST_URL = os.getenv("INCIDENT_ANALYST_URL")
INCIDENT_ANALYST_TOKEN = os.getenv("INCIDENT_ANALYST_TOKEN")

print("INCIDENT_ANALYST_URL:", INCIDENT_ANALYST_URL)

In [0]:
def call_incident_analyst(question: str, line_id: str):
    if not INCIDENT_ANALYST_URL or not INCIDENT_ANALYST_TOKEN:
        raise RuntimeError("INCIDENT_ANALYST_URL or INCIDENT_ANALYST_TOKEN is not set.")

    context = build_incident_context(line_id)
    payload = {
        "question": question,
        "context": context,
    }

    headers = {
        "Authorization": f"Bearer {INCIDENT_ANALYST_TOKEN}",
        "Content-Type": "application/json",
    }

    resp = requests.post(INCIDENT_ANALYST_URL, headers=headers, json=payload)
    resp.raise_for_status()
    return resp.json()

## Step 3 – Test the Copilot

Ask a question about your line, using real context from your Gold and incidents tables.

In [0]:
test_line_id = "LINE_01"
question = "Why is BAD rate increasing on my line, and what checks should I run next?"

response = call_incident_analyst(question, test_line_id)
print(json.dumps(response, indent=2))